# Data Quality and Reconciliation

The requirement is that 100% of the provided data is ingested. This notebook
produces the numbers that prove it instead of asserting it, and fails the
Lakeflow job if any of them drift.

In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG_NAME = 'beverage_sales'

UNKNOWN_KEY = -1

## Row count and dollar volume across the layers

In [0]:
df_reconciliation = spark.sql(f"""
SELECT 'bronze' AS layer,
       count(*) AS row_count,
       sum(cast(dollar_volume AS decimal(18,2))) AS dollar_volume
FROM {CATALOG_NAME}.bronze.sales
UNION ALL
SELECT 'silver', count(*), sum(dollar_volume)
FROM {CATALOG_NAME}.silver.sales
UNION ALL
SELECT 'gold', count(*), sum(dollar_volume)
FROM {CATALOG_NAME}.gold.fact_sales
""")

display(df_reconciliation)

## Referential integrity and duplicates

In [0]:
df_integrity = spark.sql(f"""
SELECT
  sum(CASE WHEN brand_key   = {UNKNOWN_KEY} THEN 1 ELSE 0 END) AS unknown_brand,
  sum(CASE WHEN region_key  = {UNKNOWN_KEY} THEN 1 ELSE 0 END) AS unknown_region,
  sum(CASE WHEN channel_key = {UNKNOWN_KEY} THEN 1 ELSE 0 END) AS unknown_channel,
  sum(CASE WHEN package_key = {UNKNOWN_KEY} THEN 1 ELSE 0 END) AS unknown_package,
  count(*) - count(DISTINCT date_key, brand_key, region_key, channel_key, package_key) AS duplicate_grain
FROM {CATALOG_NAME}.gold.fact_sales
""")

display(df_integrity)

## Derived calendar against the period supplied in the file

In [0]:
df_period_check = spark.sql(f"""
SELECT count(*) AS period_mismatches
FROM {CATALOG_NAME}.gold.fact_sales fs
JOIN {CATALOG_NAME}.gold.dim_date dd
  ON fs.date_key = dd.date_key
WHERE fs.period <> dd.period
""")

display(df_period_check)

## Validations

These compare the layers against each other and hold for any extract.

In [0]:
reconciliation = df_reconciliation.collect()

row_counts = {row['layer']: row['row_count'] for row in reconciliation}
dollar_volumes = {row['layer']: float(row['dollar_volume']) for row in reconciliation}
integrity = df_integrity.first()

assert row_counts['silver'] == row_counts['bronze'], 'Rows were lost between Bronze and Silver'
assert row_counts['gold'] == row_counts['silver'], 'Rows were lost between Silver and Gold'

assert round(dollar_volumes['silver'], 2) == round(dollar_volumes['bronze'], 2), \
    'Dollar volume does not reconcile between Bronze and Silver'
assert round(dollar_volumes['gold'], 2) == round(dollar_volumes['silver'], 2), \
    'Dollar volume does not reconcile between Silver and Gold'

assert integrity['unknown_brand'] == 0, 'Facts pointing at the unknown brand member'
assert integrity['unknown_region'] == 0, 'Facts pointing at the unknown region member'
assert integrity['unknown_channel'] == 0, 'Facts pointing at the unknown channel member'
assert integrity['unknown_package'] == 0, 'Facts pointing at the unknown package member'
assert integrity['duplicate_grain'] == 0, 'The fact grain is not unique'

assert df_period_check.first()['period_mismatches'] == 0, 'Derived period does not match the source'

print('Reusable data quality checks passed')